[properties of loci/alerts](https://antares.noirlab.edu/properties)

important loci properties:
- feature_linear_trend_magn_r (linar trend not including obs err)
- feature_linear_fit_slope_magn_r (line fit slope)
- feature_linear_trend_noise_magn_r (noise against line fit not including obs err)
- feature_linear_trend_sigma_magn_r (err of line fit not including obs err)
- feature_standard_deviation_magn_r (r std of mag sample)
- feature_linear_fit_reduced_chi2_magn_r (line fit quality)

In [18]:
from antares_devkit.models import BaseFilter
from antares_devkit.models import DevKitLocus
from antares_client import search
import numpy as np
import astropy.coordinates as coord
from astropy.time import Time

t_now = Time.now()

In [4]:
client_locus = search.get_by_id("ANT2020alvpw") # 2
locus_dict = client_locus.to_devkit()

In [11]:
client_locus.properties['feature_linear_fit_slope_magn_r']

1.0119751048769496e-05

In [17]:
ids = search.get_random_locus_ids(10)
ids

['ANT2020ar6ys',
 'ANT2020asaui',
 'ANT2020cjdq4',
 'ANT2020jfrfi',
 'ANT2026w7c16i6gv40l',
 'ANT2020e4mcm',
 'ANT2020asb2y',
 'ANT2020bur3y',
 'ANT2026c7qokfcn3vmz',
 'ANT2020b7vtq']

In [ ]:
for i in ids:
    rmjd = []
    rmag = []
    rerr = []
    locus = search.get_by_id(i) #just to ensure oids are lsst dias
        
    slope = locus.properties['feature_linear_fit_slope_magn_r']

    if t_now - locus.properties['newest_alert_observation_time'] > 7:
        print(f'{i} has no alerts within last week')
        continue
    elif locus.properties['newest_alert_magnitude'] < 21:
        print(f'{i} has mag under 21')
        continue

    for alert in locus.alerts:
         if alert.properties['ant_passband'] == 'R':                
            rmjd.append(alert.properties['ztf_jd'])
            rmag.append(alert.properties['ant_mag'])
            rerr.append(alert.properties['ant_magerr'])
    
    mjd_i = 3
    while t_now - rmjd[mjd_i] < 7:
        fitted_line = fit(line_init, rmag[0:i]["ant_mag"], photometry[0:i]["ant_mjd"])
         slope = fitted_line.slope
         if abs(slope) < slope_change_threshold:
             # no High variability
             mjd_i +=1
         else:
             return photometry
     elif (mjd_now - photometry[i]["ant_mjd"]) > time_pass_threshold:
         # no Recent variability
         return 0

